In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import os 

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [ ]:
# Sometimes Claude decides not to use a web tool if the answer to the prompt you give it is within its training data. 
# So put in a specific question, increase number of allowed domains, or just tell Claude to use the web search tool in a system prompt

messages = []
add_user_message(
    messages,
    """
    "What does the latest 2025 NIH research say about resistance training protocols for hypertrophy in older adults?"
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_01MrQUha6Lrw4pwfdGhmQmtS', container=None, content=[ServerToolUseBlock(id='srvtoolu_012tfzWwWarH7hBvaU97PZMx', caller=None, input={'query': 'NIH 2025 resistance training hypertrophy older adults'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='Eu0hCioIDxgCIiQ1ZDBhNjljZi1iZWI4LTRkZjctOTViMi1iM2ZmYjIxMWIyNGUSDG94i8EmCsdkEstFbBoMr/+dMWwgdCSivKFZIjBd3FqTYjvjXE22P5WMrPEf89lR+y6hnKefwzDWPqcMdpybsvp/8ZALNSXWDeULbPMq8CCZzSq7L5k0g9XNnYHnx5DQYwiH9JCIl12GSNlL2a4rf3b0dQJcLDRtAbz117WA4TZ4yIPszstiKg2+oLFx1e8f9BxYTvkdXU+HlESH4fToOFpXfwb/C+jGtA0b+E5hP6S0ijesHXhr28A4QoBhL9//3o70Alk5kuZgvbv/gNUlBHlAwE5LQxHVItBkpAvykpZA8aJd91oeVzGGzRfXoeYZFqLj4yVvcnEsZ9TBND0HP3lTWbZVet20QfiXpvcSGLAeva4G9lX1hZZZRIaeszWoV/LYHTk3+8UzI2/igRn6LmPP7BHsFuarpMFrs08cio5LC38uYWe5/xbQ1AkHEPTE5oDMkcsZvacPtzKYX0JofDooQjc0fF94X2x+TtvWvOyZFXFeHtdZgniUXYC2jDGGb+Nqi4aKWFcI2v3lgV1Gr+RHSLmZ4NaWOZf32V1+dFVLpghfzVf4MaaK